# 1103 43일차

## 1. furiosa_llm

서버를 띄우지 않고 파이썬 코드 안에서 모델을 NPU에 올려 추론하는 방법

`serve`는 HTTP로 주고받아서 서버와 클라이언트를 따로 띄워야 함. 한 스크립트에서 끝낼 때는 `LLM` 객체로 바로 올려 씀

1. `pip install furiosa-llm` : 설치
2. `LLM("furiosa-ai/Meta-Llama-3.1-8B-Instruct-FP8-dynamic")` : 모델을 NPU에 올림
3. `SamplingParams(max_tokens=1000, top_p=0.3, top_k=100)` : 생성 옵션을 묶어 둠
4. `llm.tokenizer.apply_chat_template(message, tokenize=False)` : 대화 목록을 모델이 읽는 형식의 문자열로 바꿈
5. `llm.generate([prompt], params)` : 추론

- `with LLM(…) as llm:`으로 열면 블록을 벗어날 때 NPU에 올린 모델을 내림
- `apply_chat_template`은 `role`·`content` 목록을 그 모델이 학습한 대화 형식 한 덩어리로 바꿈. `tokenize=False`는 토큰 번호가 아니라 문자열로 받겠다는 뜻
- `generate`는 프롬프트 목록을 받아 여러 개를 한 번에 처리함. 결과도 목록이라 `response[0].outputs[0].text`로 꺼냄
- `serve`와 달리 스크립트를 돌릴 때마다 모델을 다시 올려서 시작이 느림

#### cf) top_p와 top_k

- 모델은 매 단계에서 모든 토큰에 확률을 매김. `top_p`·`top_k`는 그중 몇 개를 후보로 남길지 자르는 값
- `top_p`는 확률 높은 순으로 더해 가다 합이 그 값에 닿으면 끊음. 확률이 한 토큰에 몰려 있으면 한두 개만 남고, 고루 퍼져 있으면 수십 개가 남음
- `top_p=0.3`은 1등 토큰 확률이 0.3을 넘으면 후보가 하나뿐이라 거의 같은 답이 나옴. `top_p=1`은 자르지 않음
- `top_k`는 확률과 상관없이 개수로 k개를 남김. 둘을 같이 주면 두 조건을 모두 통과한 토큰만 후보가 됨
- `temperature`는 확률 분포를 평평하거나 뾰족하게 바꾸고, `top_p`·`top_k`는 분포는 그대로 두고 꼬리를 잘라냄

## 2. Tool Calling

모델이 직접 답하지 않고, 어떤 함수를 어떤 인자로 부를지 골라서 돌려주는 것

날씨나 현재 시각처럼 모델 안에 없는 값은 외부 함수가 알아야 함. 모델은 함수와 인자만 정하고 실행은 코드가 함

1. `tools` : 함수 이름·설명·파라미터를 JSON 스키마로 적은 목록
2. `tool_choice={"type":"function", "function":{"name":"get_weather"}}` : 쓸 함수를 지정
3. `response.choices[0].message.tool_calls[0].function` : 모델이 고른 함수와 인자
4. `get_weather(**json.loads(tool_call.arguments))` : 인자를 풀어 실제 함수를 실행

- `description`은 모델이 함수를 고르는 근거라서 사람에게 설명하듯 적음
- `parameters`는 JSON 스키마. `properties`에 인자별 타입, `required`에 빠지면 안 되는 인자, `enum`으로 값 후보를 제한
- `arguments`는 문자열로 와서 `json.loads`로 딕셔너리로 바꿔야 함수에 넘길 수 있음
- `tool_choice`를 지정하면 고르는 단계를 건너뛰고 그 함수를 쓰게 함. 생략하면 모델이 직접 고르거나 함수를 안 쓰고 답함
- `temperature=0.0`이면 같은 질문에 같은 함수·인자가 나와서 결과를 확인하기 쉬움
- **모델은 함수를 실행하지 않음.** 실행하고 그 결과를 다시 넘기는 것은 코드가 하는 일

## 3. fxb

모델을 NPU 전용 실행 파일로 미리 컴파일해 두고, 그 파일로 모델을 올리는 것

올릴 때마다 컴파일하면 기동까지 오래 걸림. 한 번 만들어 두면 다음부터는 컴파일을 건너뛰고 다른 NPU 서버에 파일만 옮겨 쓸 수 있음

1. `sudo apt-get install --no-install-recommends gcc-aarch64-linux-gnu libc6-dev-arm64-cross` : 컴파일에 필요한 도구
2. `fxb build Qwen/Qwen3-8B-FP8 qwen3-8b-fp8` : 모델을 `.fxb` 파일로 컴파일
3. `fxb cache ls` : 만들어 둔 목록 확인
4. `fxb add ./qwen3-8b-fp8.fxb` : 캐시에 등록
5. `furiosa-llm serve Qwen/Qwen3-8B-FP8 --fxb ./qwen3-8b-fp8.fxb` : 컴파일해 둔 파일로 서버를 띄움

```python
llm = LLM("Qwen/Qwen3-8B-FP8", fxb="/root/qwen3-8b-fp8.fxb")
```

- `fxb build`의 두 번째 인자는 결과 파일 이름이라 내가 정함
- `gcc-aarch64-linux-gnu`는 arm64용 컴파일러. 지금 돌리는 기계와 다른 아키텍처용 코드를 만들어야 해서 교차 컴파일 도구가 필요함
- `--no-install-recommends`는 권장 패키지까지 끌어오지 않고 적어 둔 것만 설치함
- `fxb=`를 줘도 모델 이름은 그대로 적음
- 경로는 절대 경로로 적어 두면 스크립트를 어느 폴더에서 돌려도 같은 파일을 가리킴
- `.fxb` 하나가 수백 MB라 용량을 차지함. `find / -name "<파일 이름>"`으로 어디에 떨어졌는지 찾음